In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
import os
import shutil
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

# SETUP & CONFIGURATION
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

try:
    import huggingface_hub.utils._progress
    import tqdm.std
    huggingface_hub.utils._progress.tqdm = tqdm.std.tqdm
except Exception:
    pass

RAW_IMAGES_FOLDER = "./Indian House Photos/"       
CLEAN_IMAGE_FOLDER = "./03_RAW_IMAGES/"   

os.makedirs(RAW_IMAGES_FOLDER, exist_ok=True)
os.makedirs(CLEAN_IMAGE_FOLDER, exist_ok=True)

print("Loading CLIP Model OFFLINE from local folder...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Hardware Acceleration: {device.upper()}")

model_id = "./local_clip_model" 
model = CLIPModel.from_pretrained(model_id).to(device)
processor = CLIPProcessor.from_pretrained(model_id)

# THE NEW 4-WAY AI GATEKEEPER PROMPTS
CATEGORIES = [
    "a photograph focusing on a single house exterior or a specific room inside a house", # Index 0: TARGET
    "a wide street view showing multiple different houses, a village lane, or a neighborhood landscape", # Index 1: JUNK
    "a portrait, selfie, or close-up photograph of a person or human face", # Index 2: NEW JUNK (People)
    "a graphic design logo, icon, scanned document, text, or blank page" # Index 3: NEW JUNK (Logos/Docs)
]

# Automatically ignores the .txt files in your folder
VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')

# MAIN AI FILTER PIPELINE
def process_images():
    print("\nStarting Image Filtering...")
    total_processed = 0
    total_saved = 0
    
    for image_name in os.listdir(RAW_IMAGES_FOLDER):
        
        if not image_name.lower().endswith(VALID_EXTENSIONS):
            continue
            
        image_path = os.path.join(RAW_IMAGES_FOLDER, image_name)
        total_processed += 1
        
        try:
            image = Image.open(image_path).convert("RGB")
            
            inputs = processor(text=CATEGORIES, images=image, return_tensors="pt", padding=True).to(device)
            
            with torch.no_grad():
                outputs = model(**inputs)
                probs = outputs.logits_per_image.softmax(dim=-1)
            
            # Grab all 4 probabilities
            prob_house = probs[0][0].item()
            prob_multiple = probs[0][1].item()
            prob_person = probs[0][2].item()
            prob_logo = probs[0][3].item()
            
            # Find out which category scored the absolute highest
            max_prob = max(prob_house, prob_multiple, prob_person, prob_logo)
            
            # RULE: It must predict "Single House" as the #1 most likely category, 
            # AND the confidence must be over 55%.
            if max_prob == prob_house and prob_house > 0.55:
                save_path = os.path.join(CLEAN_IMAGE_FOLDER, image_name)
                
                shutil.copy(image_path, save_path)
                total_saved += 1
                
                print(f"  [+] Kept (House): {image_name} (Confidence: {prob_house:.1%})")
            else:
                # Figure out exactly why it was dropped for the logs
                if max_prob == prob_multiple:
                    reason = "Multiple Houses / Street View"
                elif max_prob == prob_person:
                    reason = "Person / Portrait Detected"
                elif max_prob == prob_logo:
                    reason = "Logo / Document Detected"
                else:
                    reason = "Low Confidence House"
                    
                print(f"  [-] Dropped ({reason}): {image_name}")
                    
        except Exception as e:
            print(f"Error reading {image_name}: {e}")

    print("\n" + "="*40)
    print("PIPELINE COMPLETE")
    print(f"Total raw images processed: {total_processed}")
    print(f"Total valid photos saved: {total_saved}")
    print(f"Removed {total_processed - total_saved} useless images!")
    print("="*40)

if __name__ == "__main__":
    process_images()

Loading CLIP Model OFFLINE from local folder...
Hardware Acceleration: CPU


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


Starting Image Filtering...
  [-] Dropped (Logo / Document Detected): 24dc845a4617d6660777bdcfd71fd72c (1).jpeg
  [-] Dropped (Logo / Document Detected): 24dc845a4617d6660777bdcfd71fd72c.jpeg
  [-] Dropped (Person / Portrait Detected): 38692ffe-441f-4368-9141-b458e7a040ea.jpg
  [+] Kept (House): 82d37049-c96d-491c-ab92-9d9d75ee1482.jpg (Confidence: 98.7%)
  [-] Dropped (Person / Portrait Detected): a7fb41ed-adde-443b-9589-941827d4989a.jpg
  [-] Dropped (Logo / Document Detected): aaron-m-varughese-608.jpg
  [-] Dropped (Person / Portrait Detected): abdullah-k-163.jpeg
  [-] Dropped (Logo / Document Detected): anton-polyakov-193.jpeg
  [-] Dropped (Person / Portrait Detected): anurag-upadhyay-778.jpeg
  [-] Dropped (Person / Portrait Detected): anusmit-sil-952.jpeg


C:\Users\Harsh Datt\anaconda3\Lib\site-packages\PIL\Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  [-] Dropped (Logo / Document Detected): apple-touch-icon.png
  [+] Kept (House): buildings-or-apartment-under-construction-with-storm-clouds-.jpg (Confidence: 98.8%)
  [-] Dropped (Logo / Document Detected): canva (1).png
  [-] Dropped (Logo / Document Detected): canva.png
  [-] Dropped (Multiple Houses / Street View): city-public-park-surrounded-by-tall-residential-buildings-wi.jpg
  [+] Kept (House): construction.jpg (Confidence: 92.6%)
  [-] Dropped (Person / Portrait Detected): d67-hunter-883.png
  [-] Dropped (Person / Portrait Detected): db75861e-c71d-4665-a1b4-9642a3152566.jpg
  [+] Kept (House): delhi-cityscape.jpg (Confidence: 91.6%)
  [-] Dropped (Person / Portrait Detected): dev-anand-273.png
  [+] Kept (House): freddy-211.jpeg (Confidence: 58.4%)
  [-] Dropped (Logo / Document Detected): free-photo-of-a-black-and-white-photo-of-a-lighthouse (1).jpeg
  [-] Dropped (Logo / Document Detected): free-photo-of-a-black-and-white-photo-of-a-lighthouse (2).jpeg
  [-] Dropped (Logo

C:\Users\Harsh Datt\anaconda3\Lib\site-packages\PIL\Image.py:3432: DecompressionBombWarning: Image size (159120000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


  [+] Kept (House): pexels-photo-17935458.jpeg (Confidence: 83.5%)
  [+] Kept (House): pexels-photo-17939435.jpeg (Confidence: 86.0%)
  [-] Dropped (Person / Portrait Detected): pexels-photo-17957788.jpeg
  [+] Kept (House): pexels-photo-18025799.jpeg (Confidence: 92.8%)
  [+] Kept (House): pexels-photo-18048181.jpeg (Confidence: 99.7%)
  [+] Kept (House): pexels-photo-18132023.jpeg (Confidence: 74.3%)
  [+] Kept (House): pexels-photo-18205631.jpeg (Confidence: 95.7%)
  [+] Kept (House): pexels-photo-18276994.jpeg (Confidence: 99.1%)
  [+] Kept (House): pexels-photo-18276996.jpeg (Confidence: 97.4%)
  [+] Kept (House): pexels-photo-18364647.jpeg (Confidence: 99.9%)
  [+] Kept (House): pexels-photo-18408538.jpeg (Confidence: 99.6%)
  [-] Dropped (Multiple Houses / Street View): pexels-photo-18522321.jpeg
  [+] Kept (House): pexels-photo-18529267.jpeg (Confidence: 99.8%)
  [+] Kept (House): pexels-photo-1862402 (1).jpeg (Confidence: 100.0%)
  [+] Kept (House): pexels-photo-1862402 (2).jp

C:\Users\Harsh Datt\anaconda3\Lib\site-packages\PIL\Image.py:3432: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


  [-] Dropped (Multiple Houses / Street View): pexels-photo-34414701.jpeg
  [+] Kept (House): pexels-photo-34567085.jpeg (Confidence: 99.9%)
  [+] Kept (House): pexels-photo-34618650.jpeg (Confidence: 99.6%)
  [+] Kept (House): pexels-photo-34641275.jpeg (Confidence: 57.6%)
  [+] Kept (House): pexels-photo-34651982.jpeg (Confidence: 86.5%)
  [-] Dropped (Person / Portrait Detected): pexels-photo-34668848.jpeg
  [+] Kept (House): pexels-photo-34668859.jpeg (Confidence: 72.2%)
  [+] Kept (House): pexels-photo-34698628.jpeg (Confidence: 99.8%)
  [+] Kept (House): pexels-photo-34750343.jpeg (Confidence: 100.0%)
  [+] Kept (House): pexels-photo-34783110.jpeg (Confidence: 95.5%)
  [+] Kept (House): pexels-photo-34791505.jpeg (Confidence: 98.4%)
  [+] Kept (House): pexels-photo-34957338.jpeg (Confidence: 97.9%)
  [+] Kept (House): pexels-photo-34962102.jpeg (Confidence: 99.7%)
  [+] Kept (House): pexels-photo-34968154.jpeg (Confidence: 78.9%)
  [+] Kept (House): pexels-photo-34968508.jpeg (Co